## Breaking Down Tasks with Prompt Chaining

Welcome! In our previous lesson, you learned the fundamentals of communicating with Claude through the Anthropic API. You mastered sending single requests, understanding response structures, and managing multi-turn conversations within a single interaction. Now, you’re ready to take the next step and unlock even more powerful ways to work with Claude.

In this lesson, you’ll discover how to design and implement multi-step workflows using Claude, with a special focus on a technique called prompt chaining. By the end, you’ll know how to break down sophisticated tasks into manageable, reliable steps and connect them together for robust AI-powered solutions.
What is a Workflow?

A workflow is a structured sequence of steps or actions designed to accomplish a specific goal. In the context of AI systems, workflows help you organize and coordinate tasks so that each step has a clear purpose, defined inputs and outputs, and measurable success criteria. There are many types of workflows—some involve a single interaction, while others may require multiple steps, validation, or branching logic. Well-designed workflows make complex processes more predictable, easier to debug, and simpler to maintain.
Prompt Chaining and Why it Matters

Prompt chaining is one specific workflow pattern where you connect multiple separate Claude calls together, with each call building upon the output of the previous one. Unlike multi-turn conversations that happen within a single session, prompt chaining involves distinct API calls that work together to solve complex problems step by step.

The power of prompt chaining lies in its reliability and modularity. Instead of asking Claude to perform multiple complex tasks in a single prompt (which can lead to inconsistent results), you break the work into focused steps where you can validate and control the output at each stage. This approach makes your AI workflows more predictable and easier to debug.
Design the Workflow Before Coding

Before writing any code, it's important to break your task into clear, manageable steps. For our example, we'll build a simple three-step workflow:

    Generate a summary about AI in healthcare, with a strict character limit (around 300 characters).
    Validate that the summary meets the character requirement.
    Translate the validated summary into Spanish, returning only the translated text.

Each step will have its own focused prompt and clear input/output, making the workflow easy to follow and debug. This approach helps ensure each part works as expected before moving to the next.
Step 1: Generate a Constrained Summary

Let's start building our chain by creating the first step: generating a summary with specific character constraints. This step demonstrates how to use system prompts and user messages effectively to get predictable output from Claude.

```Python
from anthropic import Anthropic

# Initialize the Anthropic client
client = Anthropic()

# Choose a model to use
model = "claude-sonnet-4-6"

# Step 1: Ask Claude to write a summary with specific character constraints
summary_prompt = "You are a helpful assistant that writes clear, concise summaries."

summary_messages = [
    {
        "role": "user", 
        "content": "Write a 300 characters summary of artificial intelligence and its current applications in healthcare."
    }
]

# Send the first request to Claude for summary generation
summary_response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=summary_messages,
    system=summary_prompt
)

# Extract the summary text from Claude's response
summary_text = summary_response.content[0].text
print("Summary:")
print(summary_text)
```

Notice how we separate the system prompt from the user message. The system prompt establishes Claude's role as a summary writer, while the user message contains the specific task and constraints. This separation makes our prompts more maintainable and allows us to reuse the same system prompt for different summary tasks.

The user message is explicit about the character requirement. Instead of saying "write a short summary", we specify exactly "300 characters" to make the constraint testable and clear. This precision is essential in prompt chaining because the output of this step becomes the input for the next step.

When you run this code, you'll see output similar to:

Plain text
Summary:
AI in healthcare revolutionizes diagnosis through medical imaging analysis, enables personalized treatment plans, automates administrative tasks, assists in drug discovery, and powers virtual health assistants for patient care and monitoring.

The summary_response.content[0].text extraction pattern should be familiar from our previous lesson. We're accessing the first content block and extracting its text content, which works well for simple text responses like this summary.
Step 2: Validate and Guardrail the Output

The second step in our chain adds a crucial validation layer that ensures our summary meets the character requirements before proceeding to translation. This validation step demonstrates how to build reliable guardrails into your prompt chains.

```Python
# Step 2: Validate that the summary meets our character requirements
if not (250 <= len(summary_text) <= 350):
    raise ValueError(f"Summary does not meet character requirement (250-350 characters). Got {len(summary_text)} characters.")

print(f"✅ SUCCESS: Summary meets character requirement: {len(summary_text)} characters")
```

This validation step uses a programmatic check rather than asking Claude to validate its own output. We define a reasonable range (250-350 characters) instead of requiring exactly 300 characters, which gives Claude some flexibility while still meeting our needs.

When the validation fails, we raise a ValueError with a descriptive message that includes both the expected range and the actual character count. This makes debugging easier when your chain encounters problems. In production systems, you might want to implement retry logic here, perhaps asking Claude to revise the summary with tighter constraints.

When the validation passes, you'll see output like:

Plain text
✅ SUCCESS: Summary meets character requirement: 298 characters

Without validation, a summary that's too long or too short could cause problems in subsequent steps. By catching and handling constraint violations early, you make your entire workflow more robust.
Step 3: Feed the Output into Translation

The third step demonstrates the core concept of prompt chaining: using the output from one Claude call as input to another. This step takes our validated summary and translates it into Spanish using a focused system prompt.

```Python
# Step 3: Chain the summary output as input to a translation task
translation_prompt = "You are a professional translator that provides accurate Spanish translations."

translation_messages = [
    {
        "role": "user",
        "content": f"Return me just the Spanish translation of the following text:\n\n{summary_text}"
    }
]

# Send the second request to Claude using the summary from the first call
translation_response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=translation_messages,
    system=translation_prompt
)

# Extract and display the final translation result
translation_text = translation_response.content[0].text
print("Spanish Translation:")
print(translation_text)
```

The system prompt for this step is focused specifically on translation rather than general assistance. This specialization helps Claude understand its role in this step of the chain and produces more consistent results.

The key insight here is how we safely pass the summary_text variable from step one into the user message for step three. We use an f-string to embed the summary directly into the prompt, creating a clear separation between our instruction ("Return me just the Spanish translation") and the content to be translated.

When you run this final step, you'll see output like:

Plain text
Spanish Translation:
La IA en salud revoluciona el diagnóstico mediante análisis de imágenes médicas, permite planes de tratamiento personalizados, automatiza tareas administrativas, ayuda en el descubrimiento de medicamentos y potencia asistentes virtuales de salud para el cuidado y monitoreo de pacientes.

Each step builds naturally on the previous one, creating a smooth workflow from English summary generation through validation to Spanish translation.
Summary & Prep for Practice

You've successfully designed and implemented a three-step prompt chain that demonstrates the core concepts of sequential AI workflows. Your chain writes a constrained summary, validates that it meets requirements, then uses that validated output as input for translation. The key patterns you've learned include decomposing complex tasks into focused steps, using validation and guardrails between steps, and safely passing output from one Claude call as input to the next.

In the upcoming practice exercises, you'll implement this code yourself and extend it with additional features. The foundation you've built with prompt chaining opens up possibilities for much more sophisticated AI workflows. As you continue through this course, you'll see how these basic chaining concepts extend to tool usage, dynamic workflows, and complex agent behaviors that can handle real-world business problems.


## Building Your First Chain Link

Now that you understand the power of prompt chaining from the lesson, it's time to build your first chain link! In this exercise, you'll implement the foundation of a prompt chain by creating the summary generation step, which will later feed into validation and translation.

Your goal is to complete the missing code that makes a successful API call to Claude for generating a constrained summary. You'll need to:

    Create a system prompt that establishes Claude's role as a summary assistant
    Build a user message requesting a 300-character summary about something
    Make the complete API call with all required parameters
    Extract the summary text from Claude's response

Pay attention to the format of the messages list, and remember to include both the system parameter and the messages parameter in your API call.

Once you complete this step, you'll have created the first reliable link in a prompt chain that produces focused, constrained output ready for the next step in the workflow!

```python
from anthropic import Anthropic

# Initialize the Anthropic client
client = Anthropic()

# Choose a model to use
model = "claude-sonnet-4-6"

# Step 1: Ask Claude to write a summary with specific character constraints
# TODO: Create a system prompt that tells Claude to be a helpful summary assistant
summary_prompt = 

# TODO: Create the messages list with a user message requesting a 300-character summary about something
summary_messages = 

# TODO: Make the API call to Claude with model, max_tokens=2000, messages, and system parameters
summary_response = 

# TODO: Extract the text from the response and assign it to summary_text
summary_text = 

print("Summary:")
print(summary_text)

```

To build the first link in your prompt chain, you need to use the Anthropic Python SDK to configure the system prompt, format the messages array, send the request, and safely extract the generated text.

Here is the complete, working Python code with all the missing pieces implemented:

```python
from anthropic import Anthropic

# Initialize the Anthropic client
client = Anthropic()

# Choose a model to use
model = "claude-sonnet-4-6"

# Step 1: Ask Claude to write a summary with specific character constraints
# TODO: Create a system prompt that tells Claude to be a helpful summary assistant
summary_prompt = "You are a helpful summary assistant. Your task is to extract the absolute core ideas of the text provided and write a highly condensed summary."

# TODO: Create the messages list with a user message requesting a 300-character summary about something
summary_messages = [
    {
        "role": "user",
        "content": (
            "Please write a concise summary of the concept of 'Prompt Chaining' in generative AI. "
            "The final summary must be under 300 characters long, including spaces."
        )
    }
]

# TODO: Make the API call to Claude with model, max_tokens=2000, messages, and system parameters
summary_response = client.messages.create(
    model=model,
    max_tokens=2000,
    system=summary_prompt,
    messages=summary_messages
)

# TODO: Extract the text from the response and assign it to summary_text
summary_text = summary_response.content[0].text

print("Summary:")
print(summary_text)

```

### 🧱 Breaking Down the Chain Link

* **`system=summary_prompt`**: This defines the overarching guardrails and behavioral identity for Claude before it parses the input data.
* **`summary_messages`**: This passes the actual task instructions. By clearly stating constraints like *"under 300 characters long"*, you ensure that the text payload generated in this link is predictable and well-formatted for downstream automation.
* **`summary_response.content[0].text`**: This safely navigates Anthropic's block structure to capture the string response, making it ready to be stored in `summary_text` so you can pass it to the next prompt link in your application workflow.

## Adding Guardrails to Your Chain

## Tightening Guardrails for Better Control

## Completing Your Translation Chain Link

## Adding Quality Review to Your Chain